# Import Library

In [1]:
import pandas as pd
import numpy as np
import requests
from scipy.stats.mstats import winsorize
from pandas import json_normalize

# Load Dataset

In [3]:
df = pd.read_csv('housing_dirty.csv')
df.head()

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


# Eksplorasi Awal

In [8]:
print("Shape : ", df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

Shape :  (130, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.000000   1991.25000

# Hapus Data Duplikat

In [9]:
df.drop_duplicates(inplace=True)

print("Jumlah duplikat setelah dibersihkan:", df.duplicated().sum())
print("Shape setelah hapus duplikat:", df.shape)

Jumlah duplikat setelah dibersihkan: 0
Shape setelah hapus duplikat: (130, 7)


# Normalisasi string

In [11]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# Imputasi missing values

In [14]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print(df.isnull().sum())

id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


# Tangani outlier dengan IQR Fence

In [16]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

    print(f"{col}: batas bawah = {lower}, batas atas = {upper}")

harga_juta: batas bawah = -422.75, batas atas = 1719.25
luas_m2: batas bawah = -145.225, batas atas = 512.9749999999999
tahun_bangun: batas bawah = 1960.5, batas atas = 2042.5
Total missing: 0
Total duplikat: 0
Shape akhir: (130, 7)


In [18]:
print("Total missing:", df.isnull().sum().sum())
print("Total duplikat:", df.duplicated().sum())
print("Shape akhir:", df.shape)

assert df.isnull().sum().sum() == 0, "Masih ada missing values!"
assert df.duplicated().sum() == 0, "Masih ada duplikat!"

Total missing: 0
Total duplikat: 0
Shape akhir: (130, 7)


# Export dataset bersih

In [19]:
df.to_csv('housing_clean.csv', index=False)

print("Dataset bersih berhasil disimpan sebagai housing_clean.csv")

Dataset bersih berhasil disimpan sebagai housing_clean.csv


# Akses API JSONPlaceholder

In [20]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url, timeout=10)

if response.status_code == 200:
    data = response.json()
    df_api = json_normalize(data, sep='_')
    display(df_api[['id', 'name', 'email', 'address_city']])
else:
    print("Error:", response.status_code)

,id,name,email,address_city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury
